# 03 - Random Forest Classifier: EEG Eye State Classification

Trains and evaluates a Random Forest classifier on the same multi-band spectral power features used in `02_knn_classifier.ipynb`,
built in `01_data_exploration.ipynb`.

Includes randomized hyperparameter search, full evaluation (accuracy, confusion matrix), feature importance ranking,
a learning curve to check for overfitting, a calibration curve to assess predicted-probability trustworthiness, and an error
distribution plot.

Run `01_data_exploration.ipynb` first to generate `eeg_features.npz`.


## Setup

In [ ]:
# !pip install mne scikit-learn matplotlib seaborn numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.calibration import calibration_curve

SEED = 42
np.random.seed(SEED)


## Load features

In [ ]:
data = np.load("eeg_features.npz", allow_pickle=True)
X, y, ch_names = data["X"], data["y"], data["ch_names"]

# Reconstruct feature names: channel x band, matching the order features were built in
bands = ["delta", "theta", "alpha", "beta"]
feature_names = [f"{ch}_{b}" for b in bands for ch in ch_names]

print("X shape:", X.shape, "| y shape:", y.shape)
print("Class counts [open, closed]:", np.bincount(y))


## Train/test split

Random Forests don't need feature scaling the way distance-based models do, so this step is simpler than in the KNN notebook.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Train class counts:", np.bincount(y_train))
print("Test class counts:", np.bincount(y_test))


## Hyperparameter tuning

Randomized search over tree count, depth, split/leaf sizes, and feature sampling strategy, with class-balanced weighting.

In [ ]:
rf_base = RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight="balanced")

param_dist = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [None, 20, 40, 60],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
}

rf_search = RandomizedSearchCV(
    rf_base,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    random_state=SEED,
    verbose=1,
)
rf_search.fit(X_train, y_train)

print("Best RF params:", rf_search.best_params_)
print("Best CV accuracy:", rf_search.best_score_)

rf_best = rf_search.best_estimator_


## Test-set evaluation

In [ ]:
y_pred_rf = rf_best.predict(X_test)
acc = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest test accuracy: {acc:.4f}")
print("\nClassification report:\n", classification_report(y_test, y_pred_rf))


In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_rf_df = pd.DataFrame(
    cm_rf,
    index=["True: Open (0)", "True: Closed (1)"],
    columns=["Pred: Open (0)", "Pred: Closed (1)"],
)
print(cm_rf_df)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_rf_df, annot=True, fmt="d", cmap="Greens")
plt.title("Confusion Matrix - Optimized Random Forest (Eyes Open vs Closed)")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.savefig("rf_confusion_matrix.png", dpi=150)
plt.show()


## Feature importance

Which channel/band combinations mattered most? Posterior/occipital alpha-band channels should dominate, consistent with where alpha rhythms are physiologically strongest.

In [ ]:
importances = rf_best.feature_importances_
indices = np.argsort(importances)[::-1]

top_k = 20
top_idx = indices[:top_k]
top_names = [feature_names[i] for i in top_idx]
top_values = importances[top_idx]

plt.figure(figsize=(8, 5))
plt.barh(top_names[::-1], top_values[::-1])
plt.xlabel("Importance")
plt.title(f"Random Forest Feature Importances (Top {top_k})")
plt.tight_layout()
plt.savefig("rf_feature_importances.png", dpi=150)
plt.show()


In [ ]:
best_params = rf_search.best_params_
depth_str = "unlimited depth" if best_params["max_depth"] is None else f"max_depth = {best_params['max_depth']}"

print(
    f"The optimized Random Forest uses {best_params['n_estimators']} trees, {depth_str}, "
    f"min_samples_split = {best_params['min_samples_split']}, min_samples_leaf = {best_params['min_samples_leaf']}, "
    f"and max_features = '{best_params['max_features']}'."
)


## Diagnostics: overfitting, calibration, and error distribution

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    rf_best, X, y, cv=3, train_sizes=np.linspace(0.1, 1.0, 8), scoring="accuracy", n_jobs=-1
)

plt.figure(figsize=(7, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), "o-", label="Training accuracy")
plt.plot(train_sizes, test_scores.mean(axis=1), "o-", label="Validation accuracy")
plt.xlabel("Training set size")
plt.ylabel("Accuracy")
plt.title("Learning Curve - Random Forest")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("rf_learning_curve.png", dpi=150)
plt.show()


In [ ]:
y_proba = rf_best.predict_proba(X_test)[:, 1]
prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)

plt.figure(figsize=(6, 5))
plt.plot(prob_pred, prob_true, "o-", label="RF")
plt.plot([0, 1], [0, 1], "k--", label="Perfect")
plt.xlabel("Predicted probability (Closed)")
plt.ylabel("True frequency")
plt.title("Calibration Curve - Random Forest")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("rf_calibration_curve.png", dpi=150)
plt.show()


In [ ]:
errors = y_pred_rf - y_test

plt.figure(figsize=(6, 4))
sns.histplot(errors, bins=30, kde=False)
plt.xlabel("Prediction error (pred - true)")
plt.ylabel("Count")
plt.title("Error Distribution - Random Forest")
plt.tight_layout()
plt.savefig("rf_error_distribution.png", dpi=150)
plt.show()
